#### IMPORTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import os
import requests
import zipfile
from io import BytesIO
import re

In [2]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


#### DOWNLOAD AND PREP DATASET

In [3]:
def download_data():
    url = "https://download.pytorch.org/tutorial/data.zip"
    if not os.path.exists("data/eng-fra.txt"):
        print("!!! dataset not found")
        print("--- downloading dataset")
        req = requests.get(url)
        with zipfile.ZipFile(BytesIO(req.content)) as zip_ref:
            zip_ref.extractall(".")
    print("--- dataset ready")

download_data()

--- dataset ready


In [4]:
# creating language base

SOS_token = 0 # start of sentence
EOS_token = 1 # end of sentence
PAD_token = 2 # padding token

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "PAD"}
        self.n_words = 3

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [5]:
def normalizeString(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

s = "Hello! How are you? I'm fine."
print(normalizeString(s))

hello ! how are you ? i m fine .


In [6]:
print("--- reading lines")
lines = open('data/eng-fra.txt', encoding='utf-8').read().strip().split('\n')
pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

# filter for short sentences for quick training
MAX_LENGTH = 1000
good_prefixes = ("i am ", "i m ", "he is", "he s ", "she is", "she s ", "you are", "you re ", "we are", "we re ", "they are", "they re ")
pairs = [p for p in pairs if len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH and p[0].startswith(good_prefixes)]

input_lang = Lang('eng')
output_lang = Lang('fra')

print("--- building vocab")
for pair in pairs:
    input_lang.addSentence(pair[0])
    output_lang.addSentence(pair[1])

print(f"+++ counted words: \neng: {input_lang.n_words}\nfra: {output_lang.n_words}")

--- reading lines
--- building vocab
+++ counted words: 
eng: 3391
fra: 4768


In [7]:
def tensorFromSentence(lang, sentence):
    indexes = [lang.word2index[word] for word in sentence.split(' ')]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

s = "i m fine"
print(tensorFromSentence(input_lang, s))

tensor([[ 3,  4, 24,  1]], device='cuda:0')


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, pairs, input_lang, output_lang):
        self.pairs = pairs
        self.input_lang = input_lang
        self.output_lang = output_lang

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        src_tensor = tensorFromSentence(self.input_lang, self.pairs[index][0]).squeeze()
        trg_tensor = tensorFromSentence(self.output_lang, self.pairs[index][1]).squeeze()
        return src_tensor, trg_tensor

PAD_IDX = PAD_token

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src_item, trg_item in batch:
        src_batch.append(src_item)
        trg_batch.append(trg_item)
    
    # pad_sequence standardizes the length required for batch processing
    src_padded = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    trg_padded = pad_sequence(trg_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_padded, trg_padded

#### BUILDING RNN MODEL ARCHITECTURE

In [9]:
class encoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(encoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)

    def forward(self, input_seq):
        # input_seq shape: (batch_size, seq_len)
        embedded = self.embedding(input_seq)
        
        # output contains the hidden states for all timesteps
        # hidden contains the final context vector
        output, hidden = self.rnn(embedded)
        return output, hidden

class decoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(decoderRNN, self).__init__()
        self.hidden_size = hidden_size
        
        self.embedding = nn.Embedding(output_size, hidden_size)
        
        # input to the RNN: the embedded word + context vector
        self.rnn = nn.RNN(hidden_size * 2, hidden_size, batch_first=True)

        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input_step, hidden, context):
        # input_step shape: (batch_size, 1) - one word at a time
        embedded = self.embedding(input_step)

        # reshape context to (batch_size, 1, hidden_size) from (1, batch_size, hidden_size)
        context_reshaped = context.permute(1, 0, 2)
        
        # concatenate embedding and context vector (batch_size, 1, 2*hidden_size)
        emb_con = torch.cat((embedded, context_reshaped), dim=2)
        
        output, hidden = self.rnn(emb_con, hidden)
        
        # decode next word
        prediction = self.out(output.squeeze(1))
        return prediction, hidden

#### BUILDING GRU ARCHITECTURE

In [10]:
class encoderGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(encoderGRU, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input_seq):
        # input_seq shape: (batch_size, seq_len)
        embedded = self.embedding(input_seq)
        
        # output contains the hidden states for all timesteps
        # hidden contains the final context vector
        output, hidden = self.gru(embedded)
        return output, hidden

class decoderGRU(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(decoderGRU, self).__init__()
        self.hidden_size = hidden_size
        
        self.embedding = nn.Embedding(output_size, hidden_size)
        
        # input to the RNN: the embedded word + context vector
        self.gru = nn.GRU(hidden_size * 2, hidden_size, batch_first=True)

        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input_step, hidden, context):
        # input_step shape: (batch_size, 1) - one word at a time
        embedded = self.embedding(input_step)

        # reshape context to (batch_size, 1, hidden_size) from (1, batch_size, hidden_size)
        context_reshaped = context.permute(1, 0, 2)
        
        # concatenate embedding and context vector (batch_size, 1, 2*hidden_size)
        emb_con = torch.cat((embedded, context_reshaped), dim=2)
        
        output, hidden = self.gru(emb_con, hidden)
        
        # decode next word
        prediction = self.out(output.squeeze(1))
        return prediction, hidden

#### TRAINING AND TEST

In [11]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()


    # encode eng sequence
    encoder_outputs, encoder_hidden = encoder(input_tensor)
    
    # the context vector
    context = encoder_hidden

    # decoder starts with SOS token and the context vector as its initial hidden state
    batch_size = input_tensor.size(0)
    decoder_input = torch.full((batch_size, 1), SOS_token, dtype=torch.long, device=device)
    decoder_hidden = context 

    loss = 0
    target_length = target_tensor.size(1)
    
    # teacher forcing: feed the target as the next input instead of decoder's own prediction
    for di in range(target_length):
        decoder_output, decoder_hidden = decoder(
            decoder_input, decoder_hidden, context
        )
        loss += criterion(decoder_output, target_tensor[:, di])
        decoder_input = target_tensor[:, di].unsqueeze(1) # next target word

    loss.backward()
    
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item()/target_length

In [12]:
def evaluate(encoder, decoder, sentence):
    # no grad as we are not training
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        
        _, encoder_hidden = encoder(input_tensor)
        context = encoder_hidden
        
        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = context
        
        decoded_words = []

        for di in range(MAX_LENGTH):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, context)
            
            topv, topi = decoder_output.data.topk(1)
            # break loop if EOS token is predicted
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])
                
            decoder_input = topi.view(1, 1)

        return ' '.join(decoded_words)

In [13]:
# init models
hidden_size = 128
encoder = encoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = decoderRNN(hidden_size, output_lang.n_words).to(device)

# batching
batch_size = 64
dataset = TranslationDataset(pairs, input_lang, output_lang)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# set optim, loss function and lr
encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)

# loss function to ignore padding
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# training loop
epochs = 10

encoder.train()
decoder.train()

for epoch in range(1, epochs + 1):
    epoch_loss = 0
    
    for batch_idx, (input_tensor, target_tensor) in enumerate(dataloader):
        
        input_tensor = input_tensor.to(device)
        target_tensor = target_tensor.to(device)
        
        # zero the gradients
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # get loss from the train function
        loss = train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        
        epoch_loss += loss
        
    avg_loss = epoch_loss/len(dataloader)
    
    if epoch % (epochs // 10) == 0:
        print(f"+++ epoch {epoch}/{epochs} | loss: {avg_loss:.4f}")

+++ epoch 1/10 | loss: 4.0963
+++ epoch 2/10 | loss: 3.2131
+++ epoch 3/10 | loss: 2.8585
+++ epoch 4/10 | loss: 2.5839
+++ epoch 5/10 | loss: 2.3782
+++ epoch 6/10 | loss: 2.1942
+++ epoch 7/10 | loss: 2.0442
+++ epoch 8/10 | loss: 1.9142
+++ epoch 9/10 | loss: 1.8214
+++ epoch 10/10 | loss: 1.7231


In [14]:
print("---translating")
for i in range(5):
    pair = random.choice(pairs)
    print(">", pair[0])
    print("=", pair[1])
    output_sentence = evaluate(encoder, decoder, pair[0])
    print("<", output_sentence)
    print("")

---translating
> you re a big softy .
= t es un gros nounours .
< je suis d sol e de vous avoir bless e . <EOS>

> he is an archeologist s assistant .
= c est un assistant en arch ologie .
< je suis d sol e de vous avoir bless e . <EOS>

> he s kind of handsome .
= il est plut t beau .
< je suis d sol e de vous avoir bless e . <EOS>

> you are free to go out .
= tu es libre de sortir .
< je suis d sol e de vous avoir bless e . <EOS>

> she s busy now so she can t talk with you .
= elle est occup e pour le moment donc elle ne peut pas vous parler .
< je suis d sol e de vous avoir bless e . <EOS>



In [15]:
# init models
hidden_size = 128
encoder = encoderGRU(input_lang.n_words, hidden_size).to(device)
decoder = decoderGRU(hidden_size, output_lang.n_words).to(device)

# batching
batch_size = 128
dataset = TranslationDataset(pairs, input_lang, output_lang)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# set optim, loss function and lr
encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)

# loss function to ignore padding
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# training loop
epochs = 100

encoder.train()
decoder.train()

for epoch in range(1, epochs + 1):
    epoch_loss = 0
    
    for batch_idx, (input_tensor, target_tensor) in enumerate(dataloader):
        
        input_tensor = input_tensor.to(device)
        target_tensor = target_tensor.to(device)
        
        # zero the gradients
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # get loss from the train function
        loss = train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        
        epoch_loss += loss
        
    avg_loss = epoch_loss/len(dataloader)
    
    if epoch % (epochs // 10) == 0:
        print(f"+++ epoch {epoch}/{epochs} | loss: {avg_loss:.4f}")

+++ epoch 10/100 | loss: 1.8079
+++ epoch 20/100 | loss: 0.9713
+++ epoch 30/100 | loss: 0.5911
+++ epoch 40/100 | loss: 0.3918
+++ epoch 50/100 | loss: 0.2744
+++ epoch 60/100 | loss: 0.2005
+++ epoch 70/100 | loss: 0.1570
+++ epoch 80/100 | loss: 0.2398
+++ epoch 90/100 | loss: 0.1061
+++ epoch 100/100 | loss: 0.0918


In [16]:
print("---translating")
for i in range(5):
    pair = random.choice(pairs)
    print(">", pair[0])
    print("=", pair[1])
    output_sentence = evaluate(encoder, decoder, pair[0])
    print("<", output_sentence)
    print("")

---translating
> we re just students .
= nous ne sommes que des tudiants .
< nous sommes en train de nous en train de nous sommes en train de mentir . <EOS>

> i m eating here .
= je mange ici .
< je me sens ici . <EOS>

> i m in the mood to talk now .
= je suis maintenant dispos discuter .
< je suis maintenant dispos e discuter . <EOS>

> he is far from happy .
= il n est vraiment pas heureux .
< il est loin d tre heureux . <EOS>

> you are taller than me .
= vous tes plus grande que moi .
< tu es plus grande que moi . <EOS>

